In [1]:
# import
import pandas as pd
import requests
from IPython.display import display
from datetime import datetime
import plotly.express as px
from wordcloud import WordCloud
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [4]:
# pipline data
from src.etl import get_flight_data

df_flights = get_flight_data()

display(df_flights.head())

,id,airline_code,flight_number,airline_name,scheduled_time,actual_time,direction,airport_code,airport_name,city_hebrew,city_english,country_hebrew,country_english,terminal,checkin_counters,checkin_zone,flight_status,flight_status_hebrew
0,1,LY,5417,EL AL ISRAEL AIRLINES,2026-06-23 15:00:00,2026-06-23 15:33:00,D,TBS,TBILISI,טביליסי,TBILISI,גיאורגיה,GEORGIA,3,78-99,D,DEPARTED,המריאה
1,2,6H,169,ISRAIR AIRLINES,2026-06-23 15:10:00,2026-06-23 15:35:00,D,TIA,TIRANA,טיראנה,TIRANA,אלבניה,ALBANIA,3,201-212,G,DEPARTED,המריאה
2,3,IZ,338,ARKIA ISRAELI AIRLINES,2026-06-23 14:55:00,2026-06-23 15:37:00,A,MXP,MILAN-MALPENSA,מילאנו,MILAN,איטליה,ITALY,3,N/A,N/A,LANDED,נחתה
3,4,IZ,2550,ARKIA ISRAELI AIRLINES,2026-06-23 14:55:00,2026-06-23 15:41:00,A,VNO,VILNIUS INTL,וילנה,VILNIUS,ליטא,LITHUANIA,3,N/A,N/A,LANDED,נחתה
4,5,EY,594,ETIHAD AIRWAYS,2026-06-23 15:05:00,2026-06-23 15:42:00,D,AUH,ABU DHABI,אבו דאבי,ABU DHABI,איחוד האמירויות הערב,UNITED ARAB EMIRATES,3,56-65,C,DEPARTED,המריאה


### EDA - Exploratory Data Analysis

In [ ]:
# create delay column:
df_flights['delay'] = (df_flights['actual_time'] - df_flights['scheduled_time']).dt.total_seconds()/60

display(df_flights.head(15))

In [ ]:
# 1.How many flight from start of the day till now?
# to get the date of today
now = datetime.now()
today = now.date()

today_flight = df_flights[
    (df_flights['actual_time'].dt.date == today) &
    (df_flights['actual_time'] <= now)
]

departures = today_flight[today_flight['direction'] == 'D']
arrivals = today_flight[today_flight['direction'] == 'A']

print(f"Today is {today} was {len(today_flight)} flights")
print(f"Today is {today} was {len(departures)} departures")
print(f"Today is {today} was {len(arrivals)} arrivals")

In [ ]:
# 2. Which flight company is popular
flight_company = today_flight['airline_name'].value_counts()
# print(flight_company.head(5))

# bar chart to display the popular flight company today:
# first convert flight_company into df for easy create plot:
plot_flight_company = flight_company.reset_index()


# top 5 airline - head(5)
top_airline = plot_flight_company.sort_values(by='count', ascending=False).head(5)

# columns names
top_airline.columns = ['airline_name', 'count']

#color palette:
my_custom_palette = ['#4E79A7', '#F28E2B', '#E15759', '#76B7B2', '#59A14F'] 

# bulid the chart:
fig = px.bar(
    top_airline, x='count', y='airline_name',orientation='h',color='airline_name',color_discrete_sequence=my_custom_palette,
    title="Number of flights by airline:", labels={'count':'Number of flights','airline_name':'Flight Company'})

fig.show()


In [ ]:
# 3. Which destinations are popular - Most Popular Destinations:
# drop empty cells, remove spaces and upper the words
df_clean = today_flight['city_english'].dropna().str.strip().str.upper()

# how many time city show up
city_counts = df_clean.value_counts()

# print the popular destinations
# print(today_flight['city_english'].unique())

# wordCloud chart:
wordcloud = WordCloud(
    width=800, height=400, 
    background_color='white', 
    colormap='viridis',
    random_state=42
).generate_from_frequencies(city_counts.to_dict())

# display wordCloud chart:
plt.figure(figsize=(15,5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Popular Destinations")
plt.show()

In [ ]:
# 4. How many of the airlines are Israeli?"
 
today_flight = today_flight.copy()

israeli_airline = ['EL AL ISRAEL AIRLINES', 'ARKIA ISRAELI AIRLINES', 'ISRAIR AIRLINES']

# new column to sort by israeli or foreign
today_flight.loc[:, 'israeli_vs_foreign'] = today_flight['airline_name'].apply(lambda x: 'Israeli' if x in israeli_airline else 'Foreign')

flight_distribution = today_flight['israeli_vs_foreign'].value_counts()

# chart:
fig = px.pie(
    values=flight_distribution.values,
    names=flight_distribution.index,
    title="Israeli vs Foreign Airlines:",
    hole=0.4,
    color=flight_distribution.index,
    color_discrete_map={'Israeli': "#A8C6F9", 'Foreign': '#EF553B'}
)

# size of chart
fig.update_layout(
    width=600,  
    height=500
)

fig.show()

In [ ]:
# check the dates of flight scheduled_time:
print(df_flights['scheduled_time'].min())
print(df_flights['scheduled_time'].max())